In [1]:
import pandas as pd
import numpy as np
import joblib

from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

In [2]:
model = joblib.load("../models/forex_random_forest.pkl")

print("✅ Model Loaded Successfully")

✅ Model Loaded Successfully


In [3]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

price.tail()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread
495,2023-01-21 15:00:00,1,EUR/USD,1.10876,1.10912,1.10822,1.10861,2403,1.1
496,2023-01-21 16:00:00,1,EUR/USD,1.10722,1.10751,1.10671,1.10739,2081,1.5
497,2023-01-21 17:00:00,1,EUR/USD,1.10730,1.10735,1.10687,1.10713,1828,1.1
498,2023-01-21 18:00:00,1,EUR/USD,1.10729,1.10775,1.10712,1.10758,3894,1.7
499,2023-01-21 19:00:00,1,EUR/USD,1.10871,1.10902,1.10829,1.10895,2615,1.4


In [4]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

price.tail()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread
495,2023-01-21 15:00:00,1,EUR/USD,1.10876,1.10912,1.10822,1.10861,2403,1.1
496,2023-01-21 16:00:00,1,EUR/USD,1.10722,1.10751,1.10671,1.10739,2081,1.5
497,2023-01-21 17:00:00,1,EUR/USD,1.10730,1.10735,1.10687,1.10713,1828,1.1
498,2023-01-21 18:00:00,1,EUR/USD,1.10729,1.10775,1.10712,1.10758,3894,1.7
499,2023-01-21 19:00:00,1,EUR/USD,1.10871,1.10902,1.10829,1.10895,2615,1.4


In [5]:
price["MA_5"] = price["Close"].rolling(5).mean()

price["MA_20"] = price["Close"].rolling(20).mean()

price["Price_Change"] = price["Close"] - price["Open"]

price["High_Low_Range"] = price["High"] - price["Low"]

price["Return"] = price["Close"].pct_change()

price["RSI"] = RSIIndicator(
    close=price["Close"],
    window=14
).rsi()

macd = MACD(close=price["Close"])

price["MACD"] = macd.macd()

price["MACD_Signal"] = macd.macd_signal()

bb = BollingerBands(
    close=price["Close"],
    window=20,
    window_dev=2
)

price["BB_Upper"] = bb.bollinger_hband()

price["BB_Middle"] = bb.bollinger_mavg()

price["BB_Lower"] = bb.bollinger_lband()

price = price.dropna().copy()

In [6]:
X = price[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Spread",
        "MA_5",
        "MA_20",
        "RSI",
        "MACD",
        "MACD_Signal",
        "BB_Upper",
        "BB_Middle",
        "BB_Lower",
        "Price_Change",
        "High_Low_Range",
        "Return"
    ]
]

In [7]:
price["Prediction"] = model.predict(X)

price[[
    "DateTime",
    "Close",
    "Prediction"
]].tail(20)

,DateTime,Close,Prediction
480,2023-01-21 00:00:00,1.11670,0
481,2023-01-21 01:00:00,1.11600,0
482,2023-01-21 02:00:00,1.11542,0
483,2023-01-21 03:00:00,1.11369,0
484,2023-01-21 04:00:00,1.11315,1
485,2023-01-21 05:00:00,1.11335,1
486,2023-01-21 06:00:00,1.11332,1
487,2023-01-21 07:00:00,1.11365,0
488,2023-01-21 08:00:00,1.11356,0
489,2023-01-21 09:00:00,1.11285,1


In [8]:
price["Trading_Signal"] = price["Prediction"].map({
    1: "BUY",
    0: "SELL"
})

trade_signals = price[[
    "DateTime",
    "PairName",
    "Close",
    "Prediction",
    "Trading_Signal"
]]

trade_signals.to_csv("../data/generated_trade_signals.csv", index=False)

print("Generated trade signals saved successfully!")

trade_signals.tail()

Generated trade signals saved successfully!


,DateTime,PairName,Close,Prediction,Trading_Signal
495,2023-01-21 15:00:00,EUR/USD,1.10861,0,SELL
496,2023-01-21 16:00:00,EUR/USD,1.10739,0,SELL
497,2023-01-21 17:00:00,EUR/USD,1.10713,0,SELL
498,2023-01-21 18:00:00,EUR/USD,1.10758,1,BUY
499,2023-01-21 19:00:00,EUR/USD,1.10895,0,SELL


In [9]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

price.tail()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread
495,2023-01-21 15:00:00,1,EUR/USD,1.10876,1.10912,1.10822,1.10861,2403,1.1
496,2023-01-21 16:00:00,1,EUR/USD,1.10722,1.10751,1.10671,1.10739,2081,1.5
497,2023-01-21 17:00:00,1,EUR/USD,1.10730,1.10735,1.10687,1.10713,1828,1.1
498,2023-01-21 18:00:00,1,EUR/USD,1.10729,1.10775,1.10712,1.10758,3894,1.7
499,2023-01-21 19:00:00,1,EUR/USD,1.10871,1.10902,1.10829,1.10895,2615,1.4


In [10]:
# Create Moving Averages
price["MA_5"] = price["Close"].rolling(window=5).mean()

price["MA_20"] = price["Close"].rolling(window=20).mean()

In [11]:
# Create Additional Features
price["Price_Change"] = price["Close"] - price["Open"]

price["High_Low_Range"] = price["High"] - price["Low"]

price["Return"] = price["Close"].pct_change()

In [12]:
# Calculate RSI

price["RSI"] = RSIIndicator(
    close=price["Close"],
    window=14
).rsi()

price[["Close","RSI"]].tail()

,Close,RSI
495,1.10861,28.186279
496,1.10739,25.076850
497,1.10713,24.457616
498,1.10758,27.781563
499,1.10895,36.886526


In [13]:
# Calculate MACD

macd = MACD(close=price["Close"])

price["MACD"] = macd.macd()

price["MACD_Signal"] = macd.macd_signal()

price["MACD_Histogram"] = macd.macd_diff()

price[[
    "Close",
    "MACD",
    "MACD_Signal"
]].tail()

,Close,MACD,MACD_Signal
495,1.10861,-0.001262,-0.000544
496,1.10739,-0.001505,-0.000737
497,1.10713,-0.001700,-0.000929
498,1.10758,-0.001796,-0.001103
499,1.10895,-0.001742,-0.001231


In [14]:
# Calculate Bollinger Bands

bb = BollingerBands(
    close=price["Close"],
    window=20,
    window_dev=2
)

price["BB_Upper"] = bb.bollinger_hband()

price["BB_Middle"] = bb.bollinger_mavg()

price["BB_Lower"] = bb.bollinger_lband()

price[[
    "Close",
    "BB_Upper",
    "BB_Middle",
    "BB_Lower"
]].tail()

,Close,BB_Upper,BB_Middle,BB_Lower
495,1.10861,1.119466,1.113773,1.108081
496,1.10739,1.119388,1.113284,1.107179
497,1.10713,1.119065,1.112755,1.106445
498,1.10758,1.118415,1.112228,1.106040
499,1.10895,1.117523,1.111777,1.106030


In [15]:
# Select the same features used during training

X = price[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Spread",
        "MA_5",
        "MA_20",
        "RSI",
        "MACD",
        "MACD_Signal",
        "BB_Upper",
        "BB_Middle",
        "BB_Lower",
        "Price_Change",
        "High_Low_Range",
        "Return"
    ]
]

In [16]:
# Generate Predictions

price["Prediction"] = model.predict(X)

price[
    [
        "DateTime",
        "Close",
        "Prediction"
    ]
].tail(10)

,DateTime,Close,Prediction
490,2023-01-21 10:00:00,1.11329,0
491,2023-01-21 11:00:00,1.11173,0
492,2023-01-21 12:00:00,1.10983,0
493,2023-01-21 13:00:00,1.10950,1
494,2023-01-21 14:00:00,1.10983,0
495,2023-01-21 15:00:00,1.10861,0
496,2023-01-21 16:00:00,1.10739,0
497,2023-01-21 17:00:00,1.10713,0
498,2023-01-21 18:00:00,1.10758,1
499,2023-01-21 19:00:00,1.10895,0


In [17]:
# Convert Prediction into Trading Signal

price["Trading_Signal"] = price["Prediction"].map({
    1: "BUY",
    0: "SELL"
})

price[
    [
        "DateTime",
        "Close",
        "Prediction",
        "Trading_Signal"
    ]
].tail(10)

,DateTime,Close,Prediction,Trading_Signal
490,2023-01-21 10:00:00,1.11329,0,SELL
491,2023-01-21 11:00:00,1.11173,0,SELL
492,2023-01-21 12:00:00,1.10983,0,SELL
493,2023-01-21 13:00:00,1.10950,1,BUY
494,2023-01-21 14:00:00,1.10983,0,SELL
495,2023-01-21 15:00:00,1.10861,0,SELL
496,2023-01-21 16:00:00,1.10739,0,SELL
497,2023-01-21 17:00:00,1.10713,0,SELL
498,2023-01-21 18:00:00,1.10758,1,BUY
499,2023-01-21 19:00:00,1.10895,0,SELL


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [19]:
trade = pd.read_csv("../data/trade_log.csv")

trade.head()

,TradeID,PairName,StrategyName,SignalType,Direction,EntryPrice,ExitPrice,EntryDateTime,ExitDateTime,PositionSize,PnL,PnL_Pips,HoldingDays,Commission,IsOpen
0,1,USD/JPY,MACD_Trend,BUY,LONG,154.16999,154.17229,2024-07-15,2024-07-17 23:45:36.000000000,100000,2300.0,23.0,2.99,7.0,False
1,2,USD/INR,MA_Cross,STRONG_SELL,SHORT,83.90144,83.90070,2024-02-08,2024-02-10 11:45:36.000000000,100000,740.0,7.4,2.49,7.0,False
2,3,USD/JPY,RL_PPO,BUY,LONG,150.56125,150.56093,2024-06-26,2024-06-28 10:19:12.000000000,100000,-320.0,-3.2,2.43,7.0,False
3,4,GBP/USD,ML_RF,SELL,SHORT,1.23636,1.23434,2024-03-26,2024-03-28 14:24:00.000000000,100000,202.0,20.2,2.60,7.0,False
4,5,AUD/USD,RL_PPO,BUY,LONG,0.66810,0.66700,2024-06-05,2024-06-06 11:02:24.000000000,100000,-110.0,-11.0,1.46,7.0,False
